# Import Libraries

In [7]:
import pandas as pd
import numpy as np
import re
import nltk
import string
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import NMF
import gradio as gr
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PIXLAPS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Dataset

In [8]:
df = pd.read_csv("E-Commerce Reviews.csv")

df = df[["Review Text","Rating"]]
df.dropna(inplace=True)

def convert_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

df["sentiment"] = df["Rating"].apply(convert_sentiment)

df.head()

,Review Text,Rating,sentiment
0,Absolutely wonderful - silky and sexy and comf...,4,positive
1,Love this dress! it's sooo pretty. i happene...,5,positive
2,I had such high hopes for this dress and reall...,3,neutral
3,"I love, love, love this jumpsuit. it's fun, fl...",5,positive
4,This shirt is very flattering to all due to th...,5,positive


# Noisy Roman Urdu Reviews

In [9]:
extra_reviews = pd.DataFrame({
    "Review Text":[
        "product bohat acha hai",
        "delivery late thi refund chahiye",
        "quality buri hai",
        "acha product but size issue hai"
    ],
    "Rating":[
        5,
        1,
        2,
        3
    ]
})

extra_reviews["sentiment"] = extra_reviews["Rating"].apply(convert_sentiment)

df = pd.concat([df, extra_reviews])

# Preprocessing

In [10]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"\d+","",text)
    text = text.translate(str.maketrans("","",string.punctuation))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df["Review Text"].apply(preprocess)

df[["Review Text","clean_text"]].head()

,Review Text,clean_text
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable
1,Love this dress! it's sooo pretty. i happene...,love dress sooo pretty happened find store im ...
2,I had such high hopes for this dress and reall...,high hopes dress really wanted work initially ...
3,"I love, love, love this jumpsuit. it's fun, fl...",love love love jumpsuit fun flirty fabulous ev...
4,This shirt is very flattering to all due to th...,shirt flattering due adjustable front tie perf...


# Train Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"], test_size=0.2, random_state=42)

# Intent Classification

In [12]:
def intent_label(text):
    if "return" in text or "refund" in text:
        return "Refund Request"
    elif "size" in text or "fit" in text:
        return "Delivery Issue"
    elif "bad" in text or "poor" in text or "disappointed" in text:
        return "Complaint"
    else:
        return "General Query"

df["intent"] = df["clean_text"].apply(intent_label)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    df["clean_text"], df["intent"], test_size=0.2, random_state=42)

vec_i = TfidfVectorizer()
X_train_i = vec_i.fit_transform(X_train_i)
X_test_i = vec_i.transform(X_test_i)

intent_model = LogisticRegression(max_iter=1000)
intent_model.fit(X_train_i,y_train_i)

intent_pred = intent_model.predict(X_test_i)

print(classification_report(y_test_i,intent_pred))
print(confusion_matrix(y_test_i,intent_pred))

                precision    recall  f1-score   support

     Complaint       0.97      0.41      0.58        80
Delivery Issue       0.98      0.98      0.98      2341
 General Query       0.94      1.00      0.97      1757
Refund Request       1.00      0.83      0.90       351

      accuracy                           0.96      4529
     macro avg       0.97      0.80      0.86      4529
  weighted avg       0.97      0.96      0.96      4529

[[  33    8   39    0]
 [   1 2291   49    0]
 [   0    5 1752    0]
 [   0   38   23  290]]


# Topic Modeling (NMF)

In [13]:
tfidf_topic = TfidfVectorizer(max_features=1000)
X_topic = tfidf_topic.fit_transform(df["clean_text"])

nmf = NMF(n_components=3, random_state=42)
nmf.fit(X_topic)

feature_names = tfidf_topic.get_feature_names_out()

for topic_idx, topic in enumerate(nmf.components_):
    print("Topic",topic_idx)
    print([feature_names[i] for i in topic.argsort()[-10:]])

Topic 0
['really', 'large', 'ordered', 'fit', 'would', 'im', 'like', 'small', 'top', 'size']
Topic 1
['comfortable', 'dresses', 'slip', 'fabric', 'wear', 'love', 'perfect', 'flattering', 'beautiful', 'dress']
Topic 2
['pants', 'sweater', 'wear', 'color', 'perfect', 'soft', 'jeans', 'comfortable', 'great', 'love']


# Gradio Interface

In [15]:
def get_topic_keywords(topic_index):
    words = feature_names[nmf.components_[topic_index].argsort()[-5:]]
    return ", ".join(words)

def predict_review(text):
    clean = preprocess(text)
    
    vec_sent = tfidf.transform([clean])
    sentiment = lr_model.predict(vec_sent)[0]
    
    vec_int = vec_i.transform([clean])
    intent = intent_model.predict(vec_int)[0]
    
    topic_vector = nmf.transform(tfidf_topic.transform([clean]))
    topic_index = np.argmax(topic_vector)
    
    topic_words = get_topic_keywords(topic_index)
    
    return sentiment, intent, topic_words

interface = gr.Interface(
    fn=predict_review,
    inputs="text",
    outputs=["text","text","text"],
    title="Customer Reviews Intelligence System"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\anyio\to_thread.py", line 63, in run_sync
    return

Created dataset file at: .gradio\flagged\dataset1.csv


Traceback (most recent call last):
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PIXLAPS\miniconda\envs\LLM\Lib\site-packages\anyio\to_thread.py", line 63, in run_sync
    return